In [1]:
import json
import re
from pathlib import Path

import srsly
import pandas as pd
from rank_bm25 import BM25Okapi

In [2]:
# =========================
# PATHS
# =========================

QUESTIONS_FILE = Path("../QandA_100.jsonl")

PATIENT_QUERIES_FILE = Path("../../Task4_Cancer_Patients/Updated_PAR_cancer/new_PAR_queries/new_cancer_train_queries.jsonl")
ARTICLE_CORPUS_FILE = Path("../../Task4_Cancer_Patients/Updated_PAR_cancer/new_cancer_corpus.jsonl")

OUTPUT_DIR = Path("hipporag_retrieval_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 5

In [3]:
questions = pd.DataFrame(list(srsly.read_jsonl(QUESTIONS_FILE)))

questions["patient_id"] = questions["patient_id"].astype(str)
questions["article_id"] = questions["article_id"].astype(str)

questions = questions[
    questions["question"].notna()
    & (questions["question"].astype(str).str.strip() != "")
    & (~questions["question"].astype(str).str.contains("error", case=False, na=False))
].copy()

questions["question_id"] = [f"q_{i:04d}" for i in range(len(questions))]

print("Usable questions:", questions.shape)
display(questions.head())

Usable questions: (106, 7)


,patient_id,article_id,question,explanation,short_answer,long_answer,question_id
0,8674405-1,29510273,In a premenopausal woman with high-risk breast...,The case report shows a patient with multifoca...,triptorelin,The case report details a 36-year-old woman wi...,q_0000
1,8674685-1,30281871,For a 45-year-old woman with type II diabetes ...,The case report shows a patient with Langerhan...,BRAF V600E mutation-targeted therapy,The case report describes a patient with Lange...,q_0001
2,8675572-1,21380941,A 67-year-old woman with a history of endometr...,The case report shows the patient was diagnose...,colonoscopy,The correct answer is colonoscopy. According t...,q_0002
3,8675577-1,25765179,For a patient with metastatic Merkel cell carc...,The case report shows a patient with metastati...,Somatostatin receptor PET scan,The case report establishes that the patient h...,q_0003
4,8675583-1,30694442,For a patient with upper tract urothelial carc...,The case report shows a patient with upper tra...,Adjuvant immunotherapy,The patient has upper tract urothelial carcino...,q_0004


In [4]:
articles = pd.DataFrame(list(srsly.read_jsonl(ARTICLE_CORPUS_FILE)))

articles["_id"] = articles["_id"].astype(str)
articles["title"] = articles.get("title", "").fillna("")
articles["text"] = articles["text"].fillna("")

articles["retrieval_text"] = articles["title"] + " " + articles["text"]

article_ids = articles["_id"].tolist()
article_docs = articles["retrieval_text"].tolist()

article_id_to_text = dict(zip(articles["_id"], articles["retrieval_text"]))
article_id_to_title = dict(zip(articles["_id"], articles["title"]))

print("Articles:", articles.shape)
display(articles.head())

Articles: (398228, 4)


,_id,title,text,retrieval_text
0,16801861,Importance of biopsy of new bone lesions in pa...,Development of destructive bone lesions in a p...,Importance of biopsy of new bone lesions in pa...
1,17261432,Survival on dialysis post-kidney transplant fa...,A substantial number of patients return to dia...,Survival on dialysis post-kidney transplant fa...
2,15473586,Pericarditis following permanent pacemaker ins...,The appearance of pericarditis following inser...,Pericarditis following permanent pacemaker ins...
3,1379294,Primary intraosseous squamous cell carcinoma a...,Primary intraosseous carcinoma (P1OC) of the j...,Primary intraosseous squamous cell carcinoma a...
4,20699517,Inflammatory myofibroblastic tumor parieto-occ...,Inflammatory myofibroblastic tumor is a divers...,Inflammatory myofibroblastic tumor parieto-occ...


In [5]:
patients = pd.DataFrame(list(srsly.read_jsonl(PATIENT_QUERIES_FILE)))

patients["_id"] = patients["_id"].astype(str)
patients["text"] = patients["text"].fillna("")
patients["retrieval_text"] = patients["text"]

patient_ids = patients["_id"].tolist()
patient_docs = patients["retrieval_text"].tolist()

patient_id_to_text = dict(zip(patients["_id"], patients["retrieval_text"]))

print("Patients:", patients.shape)
display(patients.head())

Patients: (54810, 3)


,_id,text,retrieval_text
0,6325026-2,A 57-year-old man was admitted to the emergenc...,A 57-year-old man was admitted to the emergenc...
1,6925471-1,A 17-year-old Syrian girl presented to our hos...,A 17-year-old Syrian girl presented to our hos...
2,5223534-1,The patient presented as a 53 years old white ...,The patient presented as a 53 years old white ...
3,6982515-2,A 53-year-old man presented with a PSA level o...,A 53-year-old man presented with a PSA level o...
4,4427095-1,A 35-year-old male presented with a swelling i...,A 35-year-old male presented with a swelling i...


In [6]:
missing_gold_articles = set(questions["article_id"]) - set(articles["_id"])
missing_gold_patients = set(questions["patient_id"]) - set(patients["_id"])

print("Missing gold articles:", len(missing_gold_articles))
print("Missing gold patients:", len(missing_gold_patients))

print("Example missing articles:", list(missing_gold_articles)[:5])
print("Example missing patients:", list(missing_gold_patients)[:5])

Missing gold articles: 0
Missing gold patients: 0
Example missing articles: []
Example missing patients: []


In [7]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.split()

In [8]:
tokenized_article_docs = [tokenize(doc) for doc in article_docs]
article_bm25 = BM25Okapi(tokenized_article_docs)

print("Built article BM25 index:", len(article_docs))

tokenized_patient_docs = [tokenize(doc) for doc in patient_docs]
patient_bm25 = BM25Okapi(tokenized_patient_docs)

print("Built patient BM25 index:", len(patient_docs))

Built article BM25 index: 398228
Built patient BM25 index: 54810


In [9]:
def bm25_retrieve(query, bm25_index, doc_ids, top_k=5):
    tokenized_query = tokenize(query)
    scores = bm25_index.get_scores(tokenized_query)

    top_indices = scores.argsort()[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "id": doc_ids[idx],
            "score": float(scores[idx])
        })

    return results

In [10]:
retrieval_results = []

for _, row in questions.iterrows():
    question_id = row["question_id"]
    question = row["question"]

    gold_article_id = row["article_id"]
    gold_patient_id = row["patient_id"]

    retrieved_articles = bm25_retrieve(
        query=question,
        bm25_index=article_bm25,
        doc_ids=article_ids,
        top_k=TOP_K
    )

    retrieved_patients = bm25_retrieve(
        query=question,
        bm25_index=patient_bm25,
        doc_ids=patient_ids,
        top_k=TOP_K
    )

    retrieval_results.append({
        "question_id": question_id,
        "question": question,
        "gold_article_id": gold_article_id,
        "gold_patient_id": gold_patient_id,
        "short_answer": row.get("short_answer", ""),
        "long_answer": row.get("long_answer", ""),
        "retrieved_articles": retrieved_articles,
        "retrieved_patients": retrieved_patients
    })

print("Retrieval complete:", len(retrieval_results))

Retrieval complete: 106


In [11]:
def recall_at_k(results, gold_key, retrieved_key):
    hits = 0

    for item in results:
        gold_id = item[gold_key]
        retrieved_ids = [x["id"] for x in item[retrieved_key]]

        if gold_id in retrieved_ids:
            hits += 1

    return hits / len(results) if results else 0

In [12]:
article_recall = recall_at_k(
    retrieval_results,
    gold_key="gold_article_id",
    retrieved_key="retrieved_articles"
)

patient_recall = recall_at_k(
    retrieval_results,
    gold_key="gold_patient_id",
    retrieved_key="retrieved_patients"
)

print(f"Article Recall@{TOP_K}: {article_recall:.3f}")
print(f"Patient Recall@{TOP_K}: {patient_recall:.3f}")

Article Recall@5: 0.208
Patient Recall@5: 0.764


In [13]:
def inspect_result(i):
    item = retrieval_results[i]

    print("Question ID:", item["question_id"])
    print("Question:", item["question"])
    print()

    print("Gold article:", item["gold_article_id"])
    print("Retrieved articles:")
    for r in item["retrieved_articles"]:
        title = article_id_to_title.get(r["id"], "")
        print(f"  {r['rank']}. {r['id']} | score={r['score']:.2f} | {title}")

    print()
    print("Gold patient:", item["gold_patient_id"])
    print("Retrieved patients:")
    for r in item["retrieved_patients"]:
        print(f"  {r['rank']}. {r['id']} | score={r['score']:.2f}")

In [14]:
inspect_result(0)

Question ID: q_0000
Question: In a premenopausal woman with high-risk breast cancer who has failed ovarian suppression with goserelin, what is the most appropriate next-line treatment strategy?

Gold article: 29510273
Retrieved articles:
  1. 21555684 | score=76.23 | Impact of body mass index on the efficacy of endocrine therapy in premenopausal patients with breast cancer: an analysis of the prospective ABCSG-12 trial.
  2. 10620455 | score=74.66 | Failure of goserelin ovarian ablation in premenopausal women with breast cancer: two case reports.
  3. 12706354 | score=72.16 | The use of gonadotrophin-releasing hormone (GnRH) agonists in early and advanced breast cancer in pre- and perimenopausal women.
  4. 21541703 | score=72.01 | Failure of ovarian ablation with goserelin in a pre-menopausal breast cancer patient resulting in pregnancy: a case report and review of the literature.
  5. 24074781 | score=70.08 | Optimal systemic therapy for premenopausal women with hormone receptor-posi

In [15]:
retrieval_output_file = OUTPUT_DIR / "bm25_PAR_retrieval_results_top5.jsonl"

srsly.write_jsonl(retrieval_output_file, retrieval_results)

print("Saved:", retrieval_output_file)

Saved: hipporag_retrieval_outputs/bm25_PAR_retrieval_results_top5.jsonl


In [16]:
rag_contexts = []

for item in retrieval_results:
    article_contexts = []

    for r in item["retrieved_articles"]:
        aid = r["id"]
        article_contexts.append({
            "rank": r["rank"],
            "article_id": aid,
            "score": r["score"],
            "title": article_id_to_title.get(aid, ""),
            "text": article_id_to_text.get(aid, "")
        })

    patient_contexts = []

    for r in item["retrieved_patients"]:
        pid = r["id"]
        patient_contexts.append({
            "rank": r["rank"],
            "patient_id": pid,
            "score": r["score"],
            "text": patient_id_to_text.get(pid, "")
        })

    rag_contexts.append({
        "question_id": item["question_id"],
        "question": item["question"],
        "gold_short_answer": item["short_answer"],
        "gold_long_answer": item["long_answer"],
        "gold_article_id": item["gold_article_id"],
        "gold_patient_id": item["gold_patient_id"],
        "retrieved_articles": article_contexts,
        "retrieved_patients": patient_contexts
    })

rag_context_file = OUTPUT_DIR / "bm25_PAR_rag_contexts_top5.jsonl"
srsly.write_jsonl(rag_context_file, rag_contexts)

print("Saved:", rag_context_file)

Saved: hipporag_retrieval_outputs/bm25_PAR_rag_contexts_top5.jsonl


In [17]:
example = rag_contexts[0]

print("QUESTION:")
print(example["question"])

print("\nGOLD SHORT ANSWER:")
print(example["gold_short_answer"])

print("\nTOP ARTICLES:")
for a in example["retrieved_articles"]:
    print(f"\nRank {a['rank']} | {a['article_id']} | {a['title']}")
    print(a["text"][:500])

print("\nTOP PATIENTS:")
for p in example["retrieved_patients"]:
    print(f"\nRank {p['rank']} | {p['patient_id']}")
    print(p["text"][:500])

QUESTION:
In a premenopausal woman with high-risk breast cancer who has failed ovarian suppression with goserelin, what is the most appropriate next-line treatment strategy?

GOLD SHORT ANSWER:
triptorelin

TOP ARTICLES:

Rank 1 | 21555684 | Impact of body mass index on the efficacy of endocrine therapy in premenopausal patients with breast cancer: an analysis of the prospective ABCSG-12 trial.
Impact of body mass index on the efficacy of endocrine therapy in premenopausal patients with breast cancer: an analysis of the prospective ABCSG-12 trial. Aromatase inhibitors are effective as endocrine treatment for patients with hormone receptor-positive breast cancer. According to the hypothesis that overweight patients have higher levels of aromatase enzyme availability, we investigated the influence of body mass index (BMI) on the efficacy of adjuvant endocrine therapy in premenopausal pat

Rank 2 | 10620455 | Failure of goserelin ovarian ablation in premenopausal women with breast cancer: